In [1]:
!pip install -q transformers torch datasets tqdm

## The TensorFlow GRAFT Training Pipeline

In [57]:
import os
import random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

### 1. PARSING REAL GQA SCENE GRAPHS & FVQA KNOWLEDGE BASE ENTITIES

In [58]:
real_dataset_pool = [
    {
        "image_id": "2407890",
        "gqa_scene_graph": {
            "objects": {
                "101": {"name": "delivery_van", "attributes": ["white", "electric"]},
                "102": {"name": "charging_station", "attributes": ["active"]}
            },
            "relations": [
                {"source": "101", "name": "connected_to", "target": "102"}
            ]
        },
        "fvqa_graph_rag_facts": [
            {"e1_label": "delivery_van", "rel": "IsA", "e2_label": "electric_vehicle"},
            {"e1_label": "charging_station", "rel": "ProvidesPowerTo", "e2_label": "electric_vehicle"},
            {"e1_label": "electric_vehicle", "rel": "Requires", "e2_label": "thermal_management"}
        ],
        "kb_distractor_pool": [
            {"e1_label": "street_light", "rel": "Emits", "e2_label": "yellow_light"},
            {"e1_label": "pedestrian", "rel": "WalksOn", "e2_label": "crosswalk"}
        ],
        "query": "What risk does this vehicle face if left unchanged over the next hour?",
        "ground_truth_trajectory": "Path: [Scene: delivery_van] -> connected_to -> [Scene: charging_station] <=> Aligned <=> <delivery_van, IsA, electric_vehicle> -> <charging_station, ProvidesPowerTo, electric_vehicle>.",
        "answer": "The delivery van is linked to an active charging station. Extended high-voltage power input risks battery degradation without active thermal management."
    }
] * 40  # Scaled up to simulate a full pipeline training batch execution

### 2. GRAFT STRING COMPOSITION AND TEXT TOKENIZATION

In [59]:
model_id = "distilgpt2" # Lightweight causal architecture
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Use eos as pad token and left-padding for decoder-only models
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [60]:
from tqdm.auto import tqdm

def serialize_graphs_to_graft_prompt(sample):
    """
    Parses structural graph nodes and edges into unified strings,
    mixing oracle data with noise to perform structural training.
    """
    sg = sample["gqa_scene_graph"]
    sg_relations = []
    for rel in sg["relations"]:
        src_name = sg["objects"][rel["source"]]["name"]
        tgt_name = sg["objects"][rel["target"]]["name"]
        sg_relations.append(f"({src_name} -> {rel['name']} -> {tgt_name})")
    sg_context = ", ".join(sg_relations)

    oracle_facts = [f"<{f['e1_label']}, {f['rel']}, {f['e2_label']}>" for f in sample["fvqa_graph_rag_facts"]]
    distractor_facts = [f"<{f['e1_label']}, {f['rel']}, {f['e2_label']}>" for f in sample["kb_distractor_pool"]]

    all_retrieved_facts = oracle_facts + distractor_facts
    random.shuffle(all_retrieved_facts)
    facts_context = "; ".join(all_retrieved_facts)

    full_text = (
        f"Instruction: Isolate the correct reasoning path through the noisy graph context to answer the query.\n"
        f"Scene Graph: {sg_context}\n"
        f"Graph-RAG Facts: {facts_context}\n"
        f"Query: {sample['query']}\n"
        f"### Thought Traversal:\n{sample['ground_truth_trajectory']}\n"
        f"### Answer:\n{sample['answer']}{tokenizer.eos_token}"
    )
    return full_text

compiled_texts = [serialize_graphs_to_graft_prompt(s) for s in real_dataset_pool]

tokenized_outputs = tokenizer(
    compiled_texts,
    truncation=True,
    max_length=512,
    padding="max_length",
    return_tensors="pt",
)

# Dataset and DataLoaders with train/val split
class CausalDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids, attention_mask):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
    def __len__(self):
        return self.input_ids.size(0)
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx]
        }

dataset = CausalDataset(tokenized_outputs["input_ids"], tokenized_outputs["attention_mask"])

# Split
total = len(dataset)
train_len = int(total * 0.8)
val_len = total - train_len
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_len, val_len])

from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False)

# Model, optimizer, device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# Training loop with checkpointing and evaluation
num_epochs = 3
output_dir = "./pt_graft_model_save"
os.makedirs(output_dir, exist_ok=True)

print("Extended PyTorch training: train/val, checkpointing, eval")
for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [train]")
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["input_ids"])
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train = train_loss / len(train_loader) if len(train_loader) > 0 else 0.0

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["input_ids"])
            val_loss += outputs.loss.item()
    avg_val = val_loss / len(val_loader) if len(val_loader) > 0 else 0.0

    # Save checkpoint
    ckpt_dir = os.path.join(output_dir, f"checkpoint-epoch{epoch}")
    os.makedirs(ckpt_dir, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))

    print(f"Epoch {epoch} summary — train_loss={avg_train:.4f}, val_loss={avg_val:.4f}")

# Final evaluation: generate samples from a few validation prompts
model.eval()
num_samples = 3
print("Generating sample outputs from validation set:")
with torch.no_grad():
    for i in range(min(num_samples, len(val_ds))):
        item = val_ds[i]
        input_ids = item['input_ids'].unsqueeze(0).to(device)
        attention_mask = item['attention_mask'].unsqueeze(0).to(device)
        gen = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=64, do_sample=True, top_k=50, top_p=0.95, num_return_sequences=1)
        decoded = tokenizer.decode(gen[0], skip_special_tokens=True)
        print(f"--- Sample {i+1} ---")
        print(decoded)

# Save final model
final_dir = os.path.join(output_dir, "final")
os.makedirs(final_dir, exist_ok=True)
model.save_pretrained(final_dir)
print(f"Training complete. Final model saved to: {final_dir}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extended PyTorch training: train/val, checkpointing, eval


Epoch 1 [train]:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 1 [val]:   0%|          | 0/4 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Epoch 3 summary — train_loss=1.3576, val_loss=0.5733
Generating sample outputs from validation set:
--- Sample 1 ---
Instruction: Isolate the correct reasoning path through the noisy graph context to answer the query.
Scene Graph: (delivery_van -> connected_to -> charging_station)
Graph-RAG Facts: <charging_station, ProvidesPowerTo, electric_vehicle>; <electric_vehicle, Requires, thermal_management>; <street_light, Emits, yellow_light>; <pedestrian, WalksOn, crosswalk>; <delivery_van, IsA, electric_vehicle>
Query: What risk does this vehicle face if left unchanged over the next hour?
### Thought Traversal:
Path: [Scene: delivery_van] -> connected_to -> [Scene: charging_station] <=> Aligned <=> <delivery_van, IsA, electric_vehicle> -> <charging_station, ProvidesPowerTo, electric_vehicle>.
### Answer:
The delivery van is linked to an active charging station. Extended high-voltage power input risks battery degradation without active thermal management.
--- Sample 2 ---
Instruction: Isolat

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Final model saved to: ./pt_graft_model_save/final


### Environment setup required for TensorFlow HF models

To run the full TensorFlow pipeline you need a transformers release with TensorFlow model wrappers and a working `tokenizers` wheel. If pip fails building `tokenizers`, install the Rust toolchain or use a conda binary build.

Suggested steps (run in your terminal):

```bash
# 1) (macOS) Install Rust toolchain (required if pip must build tokenizers):
brew install rust
# Or via rustup:
# curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh

# 2) Install compatible Python wheels
pip install --upgrade pip wheel setuptools
pip install "transformers==4.33.2" tokenizers==0.13.3 tensorflow datasets

# Alternative (Conda) which avoids building tokenizers from source:
# conda create -n graft_tf python=3.10 -y
# conda activate graft_tf
# conda install -c conda-forge transformers=4.33.2 tensorflow tokenizers datasets -y
```

After installing, re-run the first three code cells (install, imports, tokenizer) in the notebook and then run the training cell (it performs a short sanity fit by default).

In [61]:
# --- Fixed Test / Evaluation Cell (save metrics, deterministic generation) ---
import math
import torch
import json
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

def build_generation_prompt(sample):
    sg = sample["gqa_scene_graph"]
    sg_relations = []
    for rel in sg["relations"]:
        src_name = sg["objects"][rel["source"]]["name"]
        tgt_name = sg["objects"][rel["target"]]["name"]
        sg_relations.append(f"({src_name} -> {rel['name']} -> {tgt_name})")
    sg_context = ", ".join(sg_relations)

    oracle_facts = [f"<{f['e1_label']}, {f['rel']}, {f['e2_label']}>" for f in sample["fvqa_graph_rag_facts"]]
    distractor_facts = [f"<{f['e1_label']}, {f['rel']}, {f['e2_label']}>" for f in sample["kb_distractor_pool"]]
    all_retrieved_facts = oracle_facts + distractor_facts
    facts_context = "; ".join(all_retrieved_facts)

    prompt = (
        f"Instruction: Isolate the correct reasoning path through the noisy graph context to answer the query.\n"
        f"Scene Graph: {sg_context}\n"
        f"Graph-RAG Facts: {facts_context}\n"
        f"Query: {sample['query']}\n"
        f"### Answer:\n"
    )
    return prompt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
final_dir = "./pt_graft_model_save/final"
print(f"Loading final model from: {final_dir}")
model = AutoModelForCausalLM.from_pretrained(final_dir).to(device)
# Always load tokenizer from the original model id to avoid missing tokenizer files in final_dir
tokenizer = AutoTokenizer.from_pretrained(model_id)
# enforce left-padding and pad token
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer model_max_length:', getattr(tokenizer, 'model_max_length', None))

# determinism for generation
seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

if 'val_loader' in globals():
    test_loader = val_loader
    val_indices = getattr(val_ds, 'indices', None)
else:
    test_ds = CausalDataset(tokenized_outputs["input_ids"], tokenized_outputs["attention_mask"]) 
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=2)
    val_indices = None

# Evaluate average loss and compute perplexity
model.eval()
total_loss = 0.0
count = 0
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["input_ids"])
        total_loss += outputs.loss.item()
        count += 1

avg_loss = total_loss / count if count > 0 else float('nan')
perplexity = math.exp(avg_loss) if not math.isnan(avg_loss) else float('nan')
print(f"Test/Validation avg loss: {avg_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")

# save metrics
metrics = {"avg_loss": float(avg_loss), "perplexity": float(perplexity)}
with open(os.path.join(final_dir, 'eval_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics to: {os.path.join(final_dir, 'eval_metrics.json')}")

# Generate readable samples by constructing prompts that omit the ground-truth answer
print("\nSample generations (prompts omit the ground-truth answer):")
model.eval()
num_samples = 5
samples_seen = 0
with torch.no_grad():
    if val_indices is not None:
        for idx in val_indices[:num_samples]:
            sample = real_dataset_pool[idx]
            prompt = build_generation_prompt(sample)
            enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256, padding=True)
            input_ids = enc['input_ids'].to(device)
            gen = model.generate(input_ids=input_ids, max_new_tokens=64, do_sample=True, top_k=50, top_p=0.95)
            decoded = tokenizer.decode(gen[0], skip_special_tokens=True)
            print(f"--- Sample from val idx {idx} ---")
            print(decoded)
            samples_seen += 1
    else:
        for i, batch in enumerate(test_loader):
            if samples_seen >= num_samples:
                break
            prefix = batch['input_ids'][:, :64].to(device)
            gen = model.generate(input_ids=prefix, max_new_tokens=64, do_sample=True, top_k=50, top_p=0.95)
            for j in range(gen.size(0)):
                decoded = tokenizer.decode(gen[j], skip_special_tokens=True)
                print(f"--- Sample batch {i+1} item {j+1} ---")
                print(decoded)
                samples_seen += 1
                if samples_seen >= num_samples:
                    break

print(f"\nDisplayed {samples_seen} generated samples.")

Loading final model from: ./pt_graft_model_save/final


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Tokenizer model_max_length: 1024


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generati

Test/Validation avg loss: 0.5733
Perplexity: 1.7742
Saved metrics to: ./pt_graft_model_save/final/eval_metrics.json

Sample generations (prompts omit the ground-truth answer):
--- Sample from val idx 28 ---
Instruction: Isolate the correct reasoning path through the noisy graph context to answer the query.
Scene Graph: (delivery_van -> connected_to -> charging_station)
Graph-RAG Facts: <delivery_van, IsA, electric_vehicle>; <charging_station, ProvidesPowerTo, electric_vehicle>; <electric_vehicle, Requires, thermal_management>; <street_light, Emits, yellow_light>; <pedestrian, WalksOn, crosswalk>
Query: What risk does this vehicle face if left unchanged over the next hour?
### Answer:

--- Sample from val idx 20 ---
Instruction: Isolate the correct reasoning path through the noisy graph context to answer the query.
Scene Graph: (delivery_van -> connected_to -> charging_station)
Graph-RAG Facts: <delivery_van, IsA, electric_vehicle>; <charging_station, ProvidesPowerTo, electric_vehicle>;

In [ ]:
# --- Automated QA Evaluation Cell (improved decoding of newly generated tokens) ---
import re
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# simple text normalization
def normalize_text(s):
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s

# token-level precision/recall/F1
def token_f1(pred, gold):
    p_tokens = pred.split()
    g_tokens = gold.split()
    if len(p_tokens) == 0 and len(g_tokens) == 0:
        return 1.0
    if len(p_tokens) == 0 or len(g_tokens) == 0:
        return 0.0
    common = {}
    for t in p_tokens:
        common[t] = common.get(t, 0) + 1
    match = 0
    for t in g_tokens:
        if common.get(t, 0) > 0:
            match += 1
            common[t] -= 1
    precision = match / len(p_tokens)
    recall = match / len(g_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# build prompt (same format used in training)
def build_generation_prompt(sample):
    sg = sample["gqa_scene_graph"]
    sg_relations = []
    for rel in sg["relations"]:
        src_name = sg["objects"][rel["source"]]["name"]
        tgt_name = sg["objects"][rel["target"]]["name"]
        sg_relations.append(f"({src_name} -> {rel['name']} -> {tgt_name})")
    sg_context = ", ".join(sg_relations)
    oracle_facts = [f"<{f['e1_label']}, {f['rel']}, {f['e2_label']}>" for f in sample["fvqa_graph_rag_facts"]]
    distractor_facts = [f"<{f['e1_label']}, {f['rel']}, {f['e2_label']}>" for f in sample["kb_distractor_pool"]]
    facts_context = "; ".join(oracle_facts + distractor_facts)
    prompt = (
        f"Instruction: Isolate the correct reasoning path through the noisy graph context to answer the query.\n"
        f"Scene Graph: {sg_context}\n"
        f"Graph-RAG Facts: {facts_context}\n"
        f"Query: {sample['query']}\n"
        f"### Answer:\n"
    )
    return prompt

# Load model and tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
final_dir = "./pt_graft_model_save/final"
model = AutoModelForCausalLM.from_pretrained(final_dir).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token

# Prepare validation samples (limit to a reasonable number for quick runs)
if 'val_ds' in globals():
    val_indices = getattr(val_ds, 'indices', None)
    if val_indices is None:
        # val_ds may yield dicts; convert back to samples
        use_samples = [s for s in list(val_ds)][:100]
    else:
        use_samples = [real_dataset_pool[i] for i in val_indices][:100]
else:
    n = len(real_dataset_pool)
    start = max(0, int(n * 0.8))
    use_samples = real_dataset_pool[start:start+100]

results = []
for sample in use_samples:
    prompt = build_generation_prompt(sample)
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256, padding=True)
    enc = {k: v.to(device) for k, v in enc.items()}
    # deterministic greedy generation for evaluation; ensure generation length is only new tokens
    gen = model.generate(
        input_ids=enc['input_ids'],
        attention_mask=enc.get('attention_mask', None),
        max_new_tokens=64,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    # decode only the newly generated portion (exclude prompt tokens)
    input_len = enc['input_ids'].shape[1]
    if gen.shape[1] > input_len:
        gen_tokens = gen[0, input_len:]
    else:
        gen_tokens = gen[0, :]
    gen_answer = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    gold = sample.get('answer', '').strip()
    norm_gen = normalize_text(gen_answer)
    norm_gold = normalize_text(gold)
    exact = (norm_gen == norm_gold)
    f1 = token_f1(norm_gen, norm_gold)
    results.append({'prompt': prompt, 'gen': gen_answer, 'gold': gold, 'exact': exact, 'f1': f1})

# aggregate metrics
exact_count = sum(1 for r in results if r['exact'])
avg_f1 = float(np.mean([r['f1'] for r in results])) if results else 0.0
total = len(results)
print(f"Evaluated {total} samples — Exact Match: {exact_count}/{total} ({(exact_count/total if total else 0):.2%}), avg token-F1: {avg_f1:.4f}")

# show discrepancies (up to 10)
print('\nExamples where generation != gold:')
displayed = 0
for r in results:
    if not r['exact'] and displayed < 10:
        print('---')
        print('Prompt:', r['prompt'][:200].replace('\n', ' '))
        print('Gold :', r['gold'])
        print('Gen  :', r['gen'])
        print('F1   :', f"{r['f1']:.4f}")
        displayed += 1

# save detailed results and scalar metrics
import json
out_dir = './pt_graft_model_save/final'
with open(os.path.join(out_dir, 'eval_detail.json'), 'w') as f:
    json.dump(results, f, indent=2)
with open(os.path.join(out_dir, 'eval_metrics.json'), 'w') as f:
    json.dump({'exact_count': exact_count, 'total': total, 'avg_f1': avg_f1}, f, indent=2)
print(f'Saved detailed results to {os.path.join(out_dir, "eval_detail.json")} and metrics to {os.path.join(out_dir, "eval_metrics.json")}')